# X-MACE Training Tutorial

**X-MACE** is a deep learning framework for modeling excited-state potential energy surfaces (PES), including conical intersections, non-adiabatic couplings (NACs), and spin-orbit couplings (SOCs). It extends the [MACE](https://github.com/ACEsuit/mace) architecture by integrating **Deep Sets** to produce smooth representations of inherently non-smooth multi-state energy surfaces.

This notebook covers:
1. Installation
2. Understanding the dataset format
3. The two model variants: **X-MACE** (AutoencoderExcitedMACE) and **E-MACE** (ExcitedMACE)
4. Training energies and forces
5. Transfer learning from a ground-state foundation model
6. Training NACs
7. Training SOCs
8. Training everything together
9. Monitoring training output

---
> **Reference:** Barrett et al., *arXiv:2502.12870* (2025). GitHub: https://github.com/rhyan10/X-MACE

## 1. Installation

X-MACE is installed by cloning the repo and checking out the `X-MACE_socs` branch. It is strongly recommended to use a dedicated conda environment.

In [1]:
# Run these commands in your terminal (not inside Python)

# --- Option A: minimal install ---
# git clone https://github.com/rhyan10/X-MACE.git
# cd X-MACE
# git checkout X-MACE_socs
# pip install .

# --- Option B: recommended (conda environment) ---
# git clone https://github.com/rhyan10/X-MACE.git
# cd X-MACE
# git checkout X-MACE_socs
# conda create --name x-mace-env python=3.13.2 -y
# conda activate x-mace-env
# pip install .

print("Installation commands printed above — run them in your terminal.")

Installation commands printed above — run them in your terminal.


## 2. Dataset Format

X-MACE reads **extended XYZ** files. Each frame's comment line stores properties as JSON arrays that ASE parses automatically into `atoms.info`.

The dataset used in this tutorial is `SINGLET_SOC_ALL.xyz`. Let's inspect what properties are available and verify their shapes.

In [4]:
import ase.io
import numpy as np

db    = ase.io.read("SINGLET_SOC_ALL.xyz", ":5")
atoms = db[0]

N_atoms  = len(atoms)                            # 23
n_states = 4
n_pairs  = n_states * (n_states - 1) // 2        # 6 unique state pairs

print(f"Atoms per frame : {N_atoms}")
print(f"Electronic states: {n_states}")
print(f"State pairs (NAC): {n_pairs}")
print()

# Quick shape summary
keys_to_check = {
    'REF_energy'      : (1, n_states),
    'REF_forces'      : (N_atoms, n_states, 3),
    'REF_smooth_nacs' : (N_atoms, n_pairs,  3),
    'REF_socs'        : (1, 240),
}
for key, expected in keys_to_check.items():
    actual = np.array(atoms.info[key]).shape
    status = "✓" if actual == expected else f"✗ got {actual}"
    print(f"  {key:15s}  expected {str(expected):15s}  {status}")

Atoms per frame : 6
Electronic states: 4
State pairs (NAC): 6

  REF_energy       expected (1, 4)           ✗ got (1, 3)
  REF_forces       expected (6, 4, 3)        ✗ got (6, 3, 3)


In [2]:
atoms.info

{'REF_energy': array([[-23090.41284093, -23090.39386643, -23090.56815697,
         -23090.44156978]]),
 'REF_forces': array([[[-1.53575034e+00, -9.81514169e+00, -5.91223068e+00],
         [-1.02048349e+01,  3.70099269e+00, -4.43855502e+00],
         [ 3.70467861e+00, -6.09539648e+00,  1.89068351e-02],
         [ 4.16167737e+00,  1.27209060e+00, -7.37033512e+00]],
 
        [[ 9.06615952e+00,  4.95021247e+00,  7.33527900e+00],
         [ 8.25126771e+00, -7.51713543e+00,  4.59413874e+00],
         [ 1.05837883e+00, -4.15851019e+00,  2.58655172e+00],
         [-5.52091280e+00, -8.71173094e+00,  7.23545418e+00]],
 
        [[-8.24630732e+00, -6.63154699e+00,  6.12474010e+00],
         [-6.76964374e+00,  7.84068021e+00, -1.09648981e+01],
         [-2.63249278e+00, -9.21826894e+00,  1.01480264e+00],
         [ 7.11257493e+00, -7.37185983e+00, -1.39234839e+00]],
 
        [[ 6.84829550e+00,  7.68363640e+00,  5.02731796e+00],
         [ 5.85404705e+00,  7.16919216e+00,  7.83253741e+00],
      

In [1]:
import torch
print("Is CUDA available?:", torch.cuda.is_available())
print("Torch version:", torch.__version__)


Is CUDA available?: False
Torch version: 2.7.1+cu118


/home/yutong/micromamba/envs/x-mace-env/lib/python3.13/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


### Key dataset fields used by X-MACE

| Field | Shape | Used for |
|---|---|---|
| `energy` | `(1, 4)` | Multi-state energy loss |
| `forces` | `(23, 4, 3)` | Per-state atomic forces |
| `smooth_nacs` / `REF_nacs` | `(23, 6, 3)` | Non-adiabatic couplings |
| `socs` | `(1, 240)` | Spin-orbit couplings (flat vector) |
| `socs_labels` | list of str | Labels to decode the SOC vector |

> **Tip:** Even if you only want to train SOCs or NACs, keep all properties in the dataset — just set the unwanted loss weights to `0.0`.

## 3. Model Variants

X-MACE provides two model types, selected via `--model`:

| Model flag | Name | Description |
|---|---|---|
| `AutoencoderExcitedMACE` | **X-MACE** | Includes a matrix diagonalisation step via an autoencoder. Better energy accuracy, especially near conical intersections. Trains energies and forces only. |
| `ExcitedMACE` | **E-MACE** | Standard multi-state readout, no diagonalisation. Faster. Required for NAC and SOC training. |

## 4. Training Energies and Forces

### 4a. X-MACE (AutoencoderExcitedMACE)

In [3]:
# Run this cell to launch training (requires a GPU and X-MACE installed)

xmace_cmd = """
python ../scripts/run_train.py \\
  --name="energies_forces" \\
  --train_file="SINGLET_SOC_ALL.xyz" \\
  --seed=100 \\
  --valid_fraction=0.1 \\
  --E0s='average' \\
  --model="AutoencoderExcitedMACE" \\
  --r_max=5.0 \\
  --batch_size=10 \\
  --n_energies=4 \\
  --correlation=3 \\
  --max_num_epochs=100 \\
  --ema \\
  --lr=0.0001 \\
  --ema_decay=0.99 \\
  --default_dtype="float32" \\
  --device=cuda \\
  --hidden_irreps="128x0e + 128x1o" \\
  --MLP_irreps='128x0e' \\
  --num_radial_basis=8 \\
  --num_interactions=2 \\
  --energy_weight=100.0 \\
  --forces_weight=100.0 \\
  --error_table="EnergyNacsDipoleMAE"
"""
print("X-MACE (Autoencoder) training command:")
print(xmace_cmd)

# Uncomment to run directly from this notebook:
import subprocess
subprocess.run(xmace_cmd, shell=True, check=True)

X-MACE (Autoencoder) training command:

python ../scripts/run_train.py \
  --name="energies_forces" \
  --train_file="SINGLET_SOC_ALL.xyz" \
  --seed=100 \
  --valid_fraction=0.1 \
  --E0s='average' \
  --model="AutoencoderExcitedMACE" \
  --r_max=5.0 \
  --batch_size=10 \
  --n_energies=4 \
  --correlation=3 \
  --max_num_epochs=100 \
  --ema \
  --lr=0.0001 \
  --ema_decay=0.99 \
  --default_dtype="float32" \
  --device=cuda \
  --hidden_irreps="128x0e + 128x1o" \
  --MLP_irreps='128x0e' \
  --num_radial_basis=8 \
  --num_interactions=2 \
  --energy_weight=100.0 \
  --forces_weight=100.0 \
  --error_table="EnergyNacsDipoleMAE"



ERROR:root:No token file found. Also make sure that a [prod] section with a 'token = value' assignment exists.
INFO:root:===========VERIFYING SETTINGS===========
INFO:root:MACE version: 0.3.6
DEBUG:root:Configuration: Namespace(name='energies_forces', seed=100, work_dir='.', nacs_key='smooth_nacs', log_dir='./logs', model_dir='.', checkpoints_dir='./checkpoints', results_dir='./results', downloads_dir='./downloads', device='cuda', default_dtype='float32', distributed=False, log_level='INFO', n_energies=4, error_table='EnergyNacsDipoleMAE', model='AutoencoderExcitedMACE', r_max=5.0, num_permutational_invariant=16, radial_type='bessel', num_radial_basis=8, num_cutoff_basis=5, pair_repulsion=False, distance_transform='None', interaction='RealAgnosticResidualInteractionBlock', interaction_first='RealAgnosticResidualInteractionBlock', max_ell=3, correlation=3, num_interactions=2, MLP_irreps='128x0e', radial_MLP='[64, 64, 64]', hidden_irreps='128x0e + 128x1o', num_channels=128, max_L=1, gate

2026-07-01 10:36:16.349 INFO: ===========VERIFYING SETTINGS===========
2026-07-01 10:36:16.349 INFO: MACE version: 0.3.6
2026-07-01 10:36:16.398 INFO: CUDA version: 11.8, CUDA device: 0
2026-07-01 10:36:16.453 INFO: 
2026-07-01 10:36:16.453 INFO: ===========LOADING INPUT DATA===========


INFO:root:Using random 10% of training set for validation with following indices: [39, 47, 4, 43, 24]
INFO:root:Atomic Numbers used: [np.int64(1)]
INFO:root:Isolated Atomic Energies (E0s) not in training file, using command line argument
INFO:root:Computing average Atomic Energies using least squares regression
INFO:root:Atomic Energies used (z: eV): {1: -1003.9330823714266}
INFO:root:
INFO:root:===========MODEL DETAILS===========


2026-07-01 10:36:16.634 INFO: Using random 10% of training set for validation with following indices: [39, 47, 4, 43, 24]
2026-07-01 10:36:16.635 WARNING: Validation batch size (10) is larger than the number of validation data (5)
2026-07-01 10:36:16.636 INFO: Atomic Numbers used: [np.int64(1)]
2026-07-01 10:36:16.636 INFO: Isolated Atomic Energies (E0s) not in training file, using command line argument
2026-07-01 10:36:16.636 INFO: Computing average Atomic Energies using least squares regression
2026-07-01 10:36:16.639 INFO: Atomic Energies used (z: eV): {1: -1003.9330823714266}
2026-07-01 10:36:16.668 INFO: 
2026-07-01 10:36:16.668 INFO: ===========MODEL DETAILS===========


...

INFO:root:Loaded Stage one model from epoch 99 for evaluation
INFO:root:Evaluating train ...
INFO:root:Evaluating valid ...
INFO:root:Error-table on TRAIN and VALID:
+-------------+----------+----------+----------+----------+
| config_type |  MAE E   |  MAE F   | MAE nacs | MAE socs |
+-------------+----------+----------+----------+----------+
|    train    |    590.3 |   3795.5 |      N/A |      N/A |
|    valid    |    515.4 |   3698.3 |      N/A |      N/A |
+-------------+----------+----------+----------+----------+
INFO:root:Saving model to checkpoints/energies_forces_run-100.model


2026-07-01 10:38:09.735 INFO: Evaluating valid ...
2026-07-01 10:38:09.871 INFO: Error-table on TRAIN and VALID:
+-------------+----------+----------+----------+----------+
| config_type |  MAE E   |  MAE F   | MAE nacs | MAE socs |
+-------------+----------+----------+----------+----------+
|    train    |    590.3 |   3795.5 |      N/A |      N/A |
|    valid    |    515.4 |   3698.3 |      N/A |      N/A |
+-------------+----------+----------+----------+----------+
2026-07-01 10:38:09.871 INFO: Saving model to checkpoints/energies_forces_run-100.model
2026-07-01 10:38:09.955 INFO: Compiling model, saving metadata to energies_forces_compiled.model


INFO:root:Compiling model, saving metadata to energies_forces_compiled.model
INFO:root:Done


2026-07-01 10:38:10.536 INFO: Done


CompletedProcess(args='\npython ../scripts/run_train.py \\\n  --name="energies_forces" \\\n  --train_file="SINGLET_SOC_ALL.xyz" \\\n  --seed=100 \\\n  --valid_fraction=0.1 \\\n  --E0s=\'average\' \\\n  --model="AutoencoderExcitedMACE" \\\n  --r_max=5.0 \\\n  --batch_size=10 \\\n  --n_energies=4 \\\n  --correlation=3 \\\n  --max_num_epochs=100 \\\n  --ema \\\n  --lr=0.0001 \\\n  --ema_decay=0.99 \\\n  --default_dtype="float32" \\\n  --device=cuda \\\n  --hidden_irreps="128x0e + 128x1o" \\\n  --MLP_irreps=\'128x0e\' \\\n  --num_radial_basis=8 \\\n  --num_interactions=2 \\\n  --energy_weight=100.0 \\\n  --forces_weight=100.0 \\\n  --error_table="EnergyNacsDipoleMAE"\n', returncode=0)

### 4b. E-MACE (ExcitedMACE) — faster baseline

A lighter model without the autoencoder diagonalisation. Use `32x0e + 32x1o` for a quick test run.

In [ ]:
emace_cmd = """
python ../scripts/run_train.py \\
  --name="energies_forces" \\
  --train_file="SINGLET_SOC_ALL.xyz" \\
  --seed=100 \\
  --valid_fraction=0.1 \\
  --E0s='average' \\
  --model="ExcitedMACE" \\
  --r_max=5.0 \\
  --batch_size=10 \\
  --n_energies=4 \\
  --correlation=3 \\
  --max_num_epochs=100 \\
  --ema \\
  --lr=0.0001 \\
  --ema_decay=0.99 \\
  --default_dtype="float32" \\
  --device=cuda \\
  --hidden_irreps="32x0e + 32x1o" \\
  --MLP_irreps='32x0e' \\
  --num_radial_basis=8 \\
  --num_interactions=2 \\
  --energy_weight=100.0 \\
  --forces_weight=1.0 \\
  --error_table="EnergyNacsDipoleMAE"
"""
print("E-MACE (no autoencoder) training command:")
print(emace_cmd)

### Key hyperparameters explained

| Flag | What it does |
|---|---|
| `--n_energies` | Number of electronic states to model (4 here) |
| `--r_max` | Cutoff radius in Å — atoms beyond this don't communicate directly |
| `--hidden_irreps` | Representation size and angular order. `128x0e + 128x1o` is large; `32x0e + 32x1o` is fast |
| `--num_interactions` | MACE message-passing layers. 2 gives receptive field of `2 × r_max` |
| `--correlation` | Body order per layer (3 = 4-body). Total order = `correlation^num_interactions` |
| `--energy_weight` / `--forces_weight` | Relative weight of energy vs force loss |
| `--ema` / `--ema_decay` | Exponential moving average of weights — stabilises training |
| `--E0s='average'` | Isolated-atom energies estimated from dataset average (convenient default) |

## 5. Transfer Learning from a Ground-State Foundation Model

X-MACE can be initialised from a pre-trained ground-state MACE model (e.g. `medium_off` from MACE-OFF23). Only the final DeepSets readout layer is reinitialised; all earlier message-passing weights are preserved. This significantly reduces the amount of excited-state data needed and improves generalisation to new molecular motifs.

In [8]:
transfer_cmd = """
python ../scripts/run_train.py \\
  --name="energies_forces_transfer" \\
  --train_file="SINGLET_SOC_ALL.xyz" \\
  --seed=100 \\
  --valid_fraction=0.1 \\
  --E0s='average' \\
  --model="ExcitedMACE" \\
  --r_max=5.0 \\
  --batch_size=10 \\
  --n_energies=4 \\
  --correlation=3 \\
  --max_num_epochs=100 \\
  --ema \\
  --lr=0.0001 \\
  --ema_decay=0.99 \\
  --default_dtype="float32" \\
  --device=cuda \\
  --hidden_irreps="128x0e + 128x1o" \\
  --MLP_irreps='128x0e' \\
  --num_radial_basis=8 \\
  --num_interactions=2 \\
  --energy_weight=100.0 \\
  --forces_weight=1.0 \\
  --error_table="EnergyNacsDipoleMAE" \\
  --foundation_model="medium_off"       # <-- key addition
"""
print("Transfer learning command:")
print(transfer_cmd)

import subprocess
subprocess.run(transfer_cmd, shell=True, check=True)

Transfer learning command:

python ../scripts/run_train.py \
  --name="energies_forces_transfer" \
  --train_file="SINGLET_SOC_ALL.xyz" \
  --seed=100 \
  --valid_fraction=0.1 \
  --E0s='average' \
  --model="ExcitedMACE" \
  --r_max=5.0 \
  --batch_size=10 \
  --n_energies=4 \
  --correlation=3 \
  --max_num_epochs=100 \
  --ema \
  --lr=0.0001 \
  --ema_decay=0.99 \
  --default_dtype="float32" \
  --device=cuda \
  --hidden_irreps="128x0e + 128x1o" \
  --MLP_irreps='128x0e' \
  --num_radial_basis=8 \
  --num_interactions=2 \
  --energy_weight=100.0 \
  --forces_weight=1.0 \
  --error_table="EnergyNacsDipoleMAE" \
  --foundation_model="medium_off"       # <-- key addition



ERROR:root:No token file found. Also make sure that a [prod] section with a 'token = value' assignment exists.
INFO:root:===========VERIFYING SETTINGS===========
INFO:root:MACE version: 0.3.6
DEBUG:root:Configuration: Namespace(name='energies_forces_transfer', seed=100, work_dir='.', nacs_key='smooth_nacs', log_dir='./logs', model_dir='.', checkpoints_dir='./checkpoints', results_dir='./results', downloads_dir='./downloads', device='cuda', default_dtype='float32', distributed=False, log_level='INFO', n_energies=4, error_table='EnergyNacsDipoleMAE', model='ExcitedMACE', r_max=5.0, num_permutational_invariant=16, radial_type='bessel', num_radial_basis=8, num_cutoff_basis=5, pair_repulsion=False, distance_transform='None', interaction='RealAgnosticResidualInteractionBlock', interaction_first='RealAgnosticResidualInteractionBlock', max_ell=3, correlation=3, num_interactions=2, MLP_irreps='128x0e', radial_MLP='[64, 64, 64]', hidden_irreps='128x0e + 128x1o', num_channels=128, max_L=1, gate='

2026-07-01 10:56:15.959 INFO: ===========VERIFYING SETTINGS===========
2026-07-01 10:56:15.959 INFO: MACE version: 0.3.6
2026-07-01 10:56:16.005 INFO: CUDA version: 11.8, CUDA device: 0
2026-07-01 10:56:16.046 INFO: Using foundation model mace-off-2023 medium as initial checkpoint. ASL license.


INFO:root:CUDA version: 11.8, CUDA device: 0
INFO:root:
INFO:root:===========LOADING INPUT DATA===========
INFO:root:Using random 10% of training set for validation with following indices: [39, 47, 4, 43, 24]
INFO:root:Atomic Numbers used: [np.int64(1)]
INFO:root:Isolated Atomic Energies (E0s) not in training file, using command line argument
INFO:root:Computing average Atomic Energies using least squares regression
INFO:root:Atomic Energies used (z: eV): {1: -1003.9330823714266}


The model is distributed under the Academic Software License (ASL) license, see https://github.com/gabor1/ASL 
 To use the model you accept the terms of the license.
ASL is based on the Gnu Public License, but does not permit commercial use
Cached MACE model to /home/yutong/.cache/mace/MACE-OFF23_medium.model
Using MACE-OFF23 MODEL for MACECalculator with /home/yutong/.cache/mace/MACE-OFF23_medium.model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
2026-07-01 10:56:19.606 INFO: CUDA version: 11.8, CUDA device: 0
2026-07-01 10:56:19.612 INFO: 
2026-07-01 10:56:19.612 INFO: ===========LOADING INPUT DATA===========
2026-07-01 10:56:19.778 INFO: Using random 10% of training set for validation with following indices: [39, 47, 4, 43, 24]
2026-07-01 10:56:19.778 WARNING: Validation batch size (10) is larger than the number of validation data (5)
2026-07-01 10:56:19.779 INFO: Atomic Numbers used: [np.int64(1)]
20

...

INFO:root:Compiling model, saving metadata to energies_forces_transfer_compiled.model


2026-07-01 10:57:54.031 INFO: Compiling model, saving metadata to energies_forces_transfer_compiled.model


INFO:root:Done


2026-07-01 10:57:54.506 INFO: Done


CompletedProcess(args='\npython ../scripts/run_train.py \\\n  --name="energies_forces_transfer" \\\n  --train_file="SINGLET_SOC_ALL.xyz" \\\n  --seed=100 \\\n  --valid_fraction=0.1 \\\n  --E0s=\'average\' \\\n  --model="ExcitedMACE" \\\n  --r_max=5.0 \\\n  --batch_size=10 \\\n  --n_energies=4 \\\n  --correlation=3 \\\n  --max_num_epochs=100 \\\n  --ema \\\n  --lr=0.0001 \\\n  --ema_decay=0.99 \\\n  --default_dtype="float32" \\\n  --device=cuda \\\n  --hidden_irreps="128x0e + 128x1o" \\\n  --MLP_irreps=\'128x0e\' \\\n  --num_radial_basis=8 \\\n  --num_interactions=2 \\\n  --energy_weight=100.0 \\\n  --forces_weight=1.0 \\\n  --error_table="EnergyNacsDipoleMAE" \\\n  --foundation_model="medium_off"       # <-- key addition\n', returncode=0)

> **Note:** `medium_off` refers to the MACE-OFF23 medium model for organic molecules. It will be downloaded automatically on first use. It is a good balance between model size and information content for organic chromophores.

## 6. Training Non-Adiabatic Couplings (NACs)

NACs are vectorial quantities (one 3-vector per atom per state pair). The number of NAC vectors is set by `--nac_num` — for 4 states this is `4×3/2 = 6`. The NAC key in the dataset is specified with `--nacs_key`.

### 6a. NACs alongside energies and forces

In [ ]:
nac_cmd = """
python scripts/run_train.py \\
  --name="nacs" \\
  --train_file="SINGLET_SOC_ALL.xyz" \\
  --seed=100 \\
  --valid_fraction=0.1 \\
  --E0s='average' \\
  --model="ExcitedMACE" \\
  --r_max=5.0 \\
  --batch_size=10 \\
  --n_energies=4 \\
  --correlation=3 \\
  --max_num_epochs=100 \\
  --ema \\
  --lr=0.0001 \\
  --ema_decay=0.99 \\
  --default_dtype="float32" \\
  --device=cuda \\
  --hidden_irreps="32x0e + 32x1o" \\
  --MLP_irreps='32x0e' \\
  --num_radial_basis=8 \\
  --num_interactions=2 \\
  --energy_weight=100.0 \\
  --forces_weight=0.0 \\
  --nacs_weight=100.0 \\
  --error_table="EnergyNacsDipoleMAE" \\
  --compute_nacs \\
  --nac_num=6 \\
  --nacs_key="REF_nacs"
"""
print("NAC training command:")
print(nac_cmd)

### 6b. NACs only (zero out other loss terms)

Keep all properties in the file but zero the loss weights for anything you don't want to train on.

In [ ]:
nac_only_cmd = """
python scripts/run_train.py \\
  --name="nacs_only" \\
  --train_file="SINGLET_SOC_ALL.xyz" \\
  --seed=100 \\
  --valid_fraction=0.1 \\
  --E0s='average' \\
  --model="ExcitedMACE" \\
  --r_max=5.0 \\
  --batch_size=10 \\
  --n_energies=4 \\
  --correlation=3 \\
  --max_num_epochs=100 \\
  --ema --lr=0.0001 --ema_decay=0.99 \\
  --default_dtype="float32" \\
  --device=cuda \\
  --hidden_irreps="32x0e + 32x1o" \\
  --MLP_irreps='32x0e' \\
  --num_radial_basis=8 \\
  --num_interactions=2 \\
  --energy_weight=0.0 \\
  --forces_weight=0.0 \\
  --nacs_weight=100.0 \\
  --socs_weight=0.0 \\
  --error_table="EnergyNacsDipoleMAE" \\
  --compute_nacs \\
  --nac_num=6 \\
  --nacs_key="REF_nacs"
"""
print("NACs-only training (all other weights = 0):")
print(nac_only_cmd)

## 7. Training Spin-Orbit Couplings (SOCs)

SOCs are scalar quantities stored as a flat vector. The number of SOC elements is set by `--soc_num`. For this dataset `soc_num=252`.

### 7a. SOCs alongside energies and forces

In [ ]:
soc_cmd = """
python scripts/run_train.py \\
  --name="socs" \\
  --train_file="SINGLET_SOC_ALL.xyz" \\
  --seed=100 \\
  --valid_fraction=0.1 \\
  --E0s='average' \\
  --model="ExcitedMACE" \\
  --r_max=5.0 \\
  --batch_size=10 \\
  --n_energies=4 \\
  --correlation=3 \\
  --max_num_epochs=100 \\
  --ema --lr=0.0001 --ema_decay=0.99 \\
  --default_dtype="float32" \\
  --device=cuda \\
  --hidden_irreps="32x0e + 32x1o" \\
  --MLP_irreps='32x0e' \\
  --num_radial_basis=8 \\
  --num_interactions=2 \\
  --energy_weight=100.0 \\
  --forces_weight=100.0 \\
  --nacs_weight=100.0 \\
  --socs_weight=100.0 \\
  --error_table="EnergyNacsDipoleMAE" \\
  --compute_socs \\
  --soc_num=252
"""
print("SOC training command:")
print(soc_cmd)

### 7b. SOCs only

In [ ]:
soc_only_cmd = """
python scripts/run_train.py \\
  --name="socs_only" \\
  --train_file="SINGLET_SOC_ALL.xyz" \\
  --seed=100 \\
  --valid_fraction=0.1 \\
  --E0s='average' \\
  --model="ExcitedMACE" \\
  --r_max=5.0 \\
  --batch_size=10 \\
  --n_energies=4 \\
  --correlation=3 \\
  --max_num_epochs=100 \\
  --ema --lr=0.0001 --ema_decay=0.99 \\
  --default_dtype="float32" \\
  --device=cuda \\
  --hidden_irreps="32x0e + 32x1o" \\
  --MLP_irreps='32x0e' \\
  --num_radial_basis=8 \\
  --num_interactions=2 \\
  --energy_weight=0.0 \\
  --forces_weight=0.0 \\
  --nacs_weight=0.0 \\
  --socs_weight=100.0 \\
  --error_table="EnergyNacsDipoleMAE" \\
  --compute_socs \\
  --soc_num=252
"""
print("SOCs-only training (all other weights = 0):")
print(soc_only_cmd)

## 8. Joint Training: Energies + Forces + NACs + SOCs

To train all quantities simultaneously, simply combine all the relevant flags. Adjust the loss weights to balance the contribution of each property.

In [ ]:
joint_cmd = """
python scripts/run_train.py \\
  --name="full_model" \\
  --train_file="SINGLET_SOC_ALL.xyz" \\
  --seed=100 \\
  --valid_fraction=0.1 \\
  --E0s='average' \\
  --model="ExcitedMACE" \\
  --r_max=5.0 \\
  --batch_size=10 \\
  --n_energies=4 \\
  --correlation=3 \\
  --max_num_epochs=100 \\
  --ema --lr=0.0001 --ema_decay=0.99 \\
  --default_dtype="float32" \\
  --device=cuda \\
  --hidden_irreps="32x0e + 32x1o" \\
  --MLP_irreps='32x0e' \\
  --num_radial_basis=8 \\
  --num_interactions=2 \\
  --energy_weight=100.0 \\
  --forces_weight=100.0 \\
  --nacs_weight=100.0 \\
  --socs_weight=100.0 \\
  --error_table="EnergyNacsDipoleMAE" \\
  --compute_nacs \\
  --nac_num=6 \\
  --nacs_key="REF_nacs" \\
  --compute_socs \\
  --soc_num=252
"""
print("Full joint training command:")
print(joint_cmd)

## 9. Monitoring Training Output

Training logs are written to the `results/` folder. The `--error_table="EnergyNacsDipoleMAE"` flag produces a per-property MAE table at each evaluation step. Let's write a small helper to parse and plot the training curves from the log file.

In [ ]:
import os
import sys
import json
from collections import defaultdict
import matplotlib.pyplot as plt


def parse_loss_log(log_path):
    """Parse training (mode='opt') and validation (mode='eval') losses from an
    X-MACE results log.

    The log is JSON-lines: one JSON object per line. Relevant keys:
        loss   -- the (weighted) loss value
        mode   -- 'opt' for per-batch training steps, 'eval' for validation
        epoch  -- integer epoch, or null for the initial pre-training eval

    Returns (train_epochs, train_loss, valid_epochs, valid_loss) where the
    per-epoch training loss is the mean of that epoch's batch losses.
    """
    if not os.path.exists(log_path):
        print(f"Log file not found: {log_path}")
        return None

    opt_by_epoch = defaultdict(list)  # epoch -> [per-batch training losses]
    valid_by_epoch = {}               # epoch -> validation loss

    with open(log_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue  # skip any non-JSON lines

            mode = rec.get("mode")
            epoch = rec.get("epoch")
            loss = rec.get("loss")
            # Skip malformed records and the initial eval (epoch is null)
            if loss is None or epoch is None:
                continue

            if mode == "opt":
                opt_by_epoch[epoch].append(loss)
            elif mode == "eval":
                valid_by_epoch[epoch] = loss

    train_epochs = sorted(opt_by_epoch)
    train_loss = [sum(opt_by_epoch[e]) / len(opt_by_epoch[e]) for e in train_epochs]

    valid_epochs = sorted(valid_by_epoch)
    valid_loss = [valid_by_epoch[e] for e in valid_epochs]

    return train_epochs, train_loss, valid_epochs, valid_loss


def main(log_path, out_path="training_curves.png"):
    result = parse_loss_log(log_path)

    if not result or not result[0]:
        print("No log data found — run a training job first.")
        return

    train_epochs, train_loss, valid_epochs, valid_loss = result

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(train_epochs, train_loss, "-o", label="Train loss (epoch mean)")
    ax.semilogy(valid_epochs, valid_loss, "-s", label="Validation loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (log scale)")
    ax.set_title("X-MACE Training Curves")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    print(f"Saved plot to {out_path}")
    plt.show()


if __name__ == "__main__":
    # Path to the X-MACE results log (JSON-lines). Default name pattern is
    # results/<name>_run-<seed>.json  e.g. results/energies_forces_run-100.json
    # Override on the command line:  python plot_loss.py results/my_run.json
    default_log = "results/energies_forces_run-100_train.txt"
    log_path = sys.argv[1] if len(sys.argv) > 1 else default_log
    main(log_path)

## 10. Quick-Reference: Flag Cheat Sheet

| Flag | Purpose | Typical value |
|---|---|---|
| `--model` | Model variant | `ExcitedMACE` / `AutoencoderExcitedMACE` |
| `--n_energies` | Number of electronic states | `4` |
| `--r_max` | Cutoff radius (Å) | `5.0` |
| `--hidden_irreps` | Representation (size × angular order) | `"128x0e + 128x1o"` (large) / `"32x0e + 32x1o"` (fast) |
| `--num_interactions` | Message-passing layers | `2` |
| `--correlation` | Body order per layer | `3` |
| `--energy_weight` | Energy contribution to loss | `100.0` |
| `--forces_weight` | Force contribution to loss | `100.0` |
| `--nacs_weight` | NAC contribution to loss | `100.0` |
| `--socs_weight` | SOC contribution to loss | `100.0` |
| `--compute_nacs` | Enable NAC head | flag (no value) |
| `--nac_num` | Number of NAC vectors | `6` for 4 states |
| `--nacs_key` | Dataset key for NACs | `"REF_nacs"` |
| `--compute_socs` | Enable SOC head | flag (no value) |
| `--soc_num` | Length of SOC flat vector | `252` |
| `--foundation_model` | Pre-trained model for transfer learning | `"medium_off"` |
| `--error_table` | Error metric to report | `"EnergyNacsDipoleMAE"` |
| `--device` | Compute device | `cuda` / `cpu` |

---

**Outputs** are written to:
- `results/` — training logs and error tables
- `checkpoints/` — model checkpoints (best model saved automatically)

**References:**
- X-MACE paper: https://arxiv.org/abs/2502.12870
- X-MACE GitHub: https://github.com/rhyan10/X-MACE
- MACE documentation: https://mace-docs.readthedocs.io